# Lab 4.5 — Enterprise Multi-Agent Platform (AI Gateway + Python Orchestrator)

**Instructor-Led Workshop · 2 Sessions (Day 1: Tasks 1–4 · Day 2: Tasks 5–9)**

### Architectural Scope & Context

This lab implements the multi-agent architecture decided in Lab 4.2:
- **Java owns ingress & governance** (Port 8080): Bearer authentication (401), RM identity (`X-RM-Id` header), rate limiting (429), and proxy failure masking (503/504).
- **Python owns orchestration & routing** (Port 8090): Deterministic keyword routing, domain specialist coordination, model fallback chain, and degradation resilience.
- **Java hosts the MCP tool bus** (`POST /mcp`, internal): Exposes `getCustomerProfile` and `getRecentTransactions` from Lab 4.4.

### Request Shape

The gateway endpoint `POST /v1/copilot/query` requires:
- Header: `Authorization: Bearer lab-token`
- Header: `X-RM-Id: rm-14`  ← the RM identity, **not** in the JSON body
- Body: `{"customerId": "10428", "prompt": "..."}` ← only these two fields

> **Note on Lab Boundaries:** Foundational components from earlier modules (Lab 4.3 security infrastructure, Lab 4.4 MCP tool definitions) are **already provided on the classpath**. Participants focus on the **9 core tasks** below.

### Validation (7 Milestones)

| Milestone | Tests |
|---|---|
| 1 — Project structure | `QueryGateway.java`, `python-orchestrator/`, `reviews/CAP_TRADEOFFS.md` present |
| 2 — Build | `mvn -q -DskipTests package` succeeds |
| 3 — Gateway proxy (Tasks 2–4) | `mvn -Dtest=GatewayAuthHttpTest test` + `QueryGatewayTest,QueryHttpTest` + `HttpOrchestratorClientTest` |
| 4 — Rules router (Task 5) | `cd python-orchestrator && pytest tests/test_router.py` |
| 5 — Specialists (Tasks 6–7) | `cd python-orchestrator && pytest tests/test_knowledge.py tests/test_agents.py` |
| 6 — End-to-end (Tasks 8–9) | `cd python-orchestrator && pytest tests/test_orchestrator.py tests/test_resilience.py` + live check |
| 7 — CAP trade-offs (Task 6) | `knowledge.py` freshness + `reviews/CAP_TRADEOFFS.md` filled |


## Day 1 — Tasks 1 to 4
---

### Task 1 — Orient: Read the Contracts
**Files to read (no code to write):**
- `INTERFACE_CONTRACT.md` — the full HTTP contract for gateway and orchestrator
- `src/main/resources/application.yml` — ports, tokens, timeouts
- `python-orchestrator/orchestrator/router.py` — the RULES table and routing order

**Key things to understand before writing any code:**
1. The RM identity travels as the `X-RM-Id` **header** — it is never in the JSON body.
2. The gateway has a 6000 ms deadline for the entire orchestrator call.
3. The router has 5 ordered rules plus a scope guardrail. `OUT_OF_SCOPE` is the default when no rule matches.
4. `POST /mcp` on the gateway has **no bearer token** — it is an internal tool bus for the Python orchestrator only.

**Verification:**
```powershell
mvn -q -DskipTests package
```
*Expect:* Build succeeds and `target/lab-4.5-multi-agent-platform-1.0.0.jar` is created.

### Task 2 — Ingress Auth (`GatewayAuthFilter.java`)
**File to edit:** `src/main/java/com/meridian/lab45/security/GatewayAuthFilter.java`

**Objective:** Implement `doFilterInternal`. Authenticate the bearer token with a constant-time comparison (`MessageDigest.isEqual`), then require a non-blank `X-RM-Id` header. A missing or wrong bearer returns 401 before spending any rate-limit token. A missing or blank `X-RM-Id` returns 400.

**What to implement inside `doFilterInternal`:**
```java
// 1. Read the Authorization header
String auth = request.getHeader("Authorization");
if (auth == null || !auth.startsWith("Bearer ")) {
    reject(response, 401, "missing token");
    return;
}
// 2. Constant-time comparison against the configured token
byte[] provided = auth.substring(7).getBytes(StandardCharsets.UTF_8);
if (!MessageDigest.isEqual(provided, expectedToken)) {
    reject(response, 401, "invalid token");
    return;
}
// 3. Require X-RM-Id
String rmId = request.getHeader("X-RM-Id");
if (rmId == null || rmId.isBlank()) {
    reject(response, 400, "X-RM-Id is required");
    return;
}
// 4. Continue
filterChain.doFilter(request, response);
```

**Verification:**
```powershell
mvn -Dtest=GatewayAuthHttpTest test
```
*Expect:* `BUILD SUCCESS`. Missing bearer → 401 `missing token`. Wrong bearer → 401 `invalid token`. Missing `X-RM-Id` → 400.

### Task 3 — Query Gateway (`QueryGateway.java`)
**File to edit:** `src/main/java/com/meridian/lab45/gateway/QueryGateway.java`

**Objective:** Implement `handle(String rmId, QueryRequest request, boolean primaryDown, boolean modelsDown)` in this order:
1. **Validate** — blank `customerId` or `prompt` → `GatewayResult.error(400, "customerId and prompt are required")` (spends no rate-limit token)
2. **Admit** — `admission.decide(rmId)`: `RM_LIMIT` → 429 `rm limit` with `Retry-After: 60`; `SHED` → 429 `shed` with `Retry-After: 1`
3. **Route** — one call: `orchestrator.route(OrchestratorCall.of(rmId, request, primaryDown, modelsDown))`
4. **Map** — the reply is already handled by the provided `map(rmId, reply)` helper

**Implementation outline:**
```java
public GatewayResult handle(String rmId, QueryRequest request,
                             boolean primaryDown, boolean modelsDown) {
    // Step 1: validate
    if (blank(request.customerId()) || blank(request.prompt())) {
        return GatewayResult.error(400, "customerId and prompt are required");
    }
    // Step 2: admit
    AdmissionControl.Decision d = admission.decide(rmId);
    if (d == AdmissionControl.Decision.RM_LIMIT) {
        return GatewayResult.tooManyRequests("rm limit", 60);
    }
    if (d == AdmissionControl.Decision.SHED) {
        return GatewayResult.tooManyRequests("shed", 1);
    }
    // Step 3 + 4: route and map
    try {
        OrchestratorReply reply = orchestrator.route(
            OrchestratorCall.of(rmId, request, primaryDown, modelsDown));
        return map(rmId, reply);
    } catch (OrchestratorUnavailableException ex) {
        log.warn("gateway rm={} orchestrator unavailable: {}", rmId, ex.getMessage());
        return GatewayResult.error(503, "orchestrator unavailable");
    } catch (OrchestratorTimeoutException ex) {
        log.warn("gateway rm={} orchestrator timeout", rmId);
        return GatewayResult.error(504, "orchestrator timeout");
    }
}
```

**Verification:**
```powershell
mvn -Dtest=QueryGatewayTest,QueryHttpTest test
```
*Expect:* `BUILD SUCCESS`. Blank body → 400. RM over limit → 429 `rm limit`. Orchestrator down → 503. Successful query → 200 with `X-Copilot-Agent` header.

### Task 4 — Orchestrator Client (`HttpOrchestratorClient.java`)
**File to edit:** `src/main/java/com/meridian/lab45/orchestrator/HttpOrchestratorClient.java`

**Objective:** Implement `route(OrchestratorCall call)`. Build an HTTP POST to `{baseUrl}/v1/orchestrate` using Java's `HttpClient`. Set headers `Content-Type: application/json`, `X-Gateway-Token`, `X-RM-Id`, and optionally `X-Lab-Primary: down` / `X-Lab-Models: down`. Translate exceptions:
- `HttpConnectTimeoutException` → `OrchestratorUnavailableException`
- `HttpTimeoutException` → `OrchestratorTimeoutException`
- `IOException` → `OrchestratorUnavailableException`

**Implementation outline:**
```java
@Override
public OrchestratorReply route(OrchestratorCall call) {
    String body = "{\"customerId\":\"" + call.customerId() + "\",\"prompt\":\"" + call.prompt() + "\"}";
    // Use mapper to serialize properly:
    // ObjectNode node = mapper.createObjectNode();
    // node.put("customerId", call.customerId()).put("prompt", call.prompt());
    // body = mapper.writeValueAsString(node);
    HttpRequest.Builder req = HttpRequest.newBuilder()
        .uri(URI.create(trimTrailingSlash(settings.baseUrl()) + "/v1/orchestrate"))
        .timeout(Duration.ofMillis(settings.deadlineMs()))
        .header("Content-Type", "application/json")
        .header("X-Gateway-Token", settings.internalToken())
        .header("X-RM-Id", call.rmId())
        .POST(HttpRequest.BodyPublishers.ofString(body, StandardCharsets.UTF_8));
    if (call.primaryDown()) req.header("X-Lab-Primary", "down");
    if (call.modelsDown()) req.header("X-Lab-Models", "down");
    try {
        HttpResponse<String> resp = http.send(req.build(), HttpResponse.BodyHandlers.ofString());
        return new OrchestratorReply(resp.statusCode(), parse(resp.body()));
    } catch (HttpConnectTimeoutException ex) {
        throw new OrchestratorUnavailableException(ex);
    } catch (HttpTimeoutException ex) {
        throw new OrchestratorTimeoutException(ex);
    } catch (IOException ex) {
        throw new OrchestratorUnavailableException(ex);
    } catch (InterruptedException ex) {
        Thread.currentThread().interrupt();
        throw new OrchestratorUnavailableException(ex);
    }
}
```

**Verification:**
```powershell
mvn -Dtest=HttpOrchestratorClientTest test
```
*Expect:* `BUILD SUCCESS`. HTTP 200 from orchestrator → `OrchestratorReply(200, body)`. Connect timeout → `OrchestratorUnavailableException`. Read timeout → `OrchestratorTimeoutException`.

## Day 2 — Tasks 5 to 9
---

### Task 5 — Rules Router (`python-orchestrator/orchestrator/router.py`)
**File to edit:** `python-orchestrator/orchestrator/router.py`

**Objective:** Implement `RulesRouter.route(self, prompt)`. Walk the `RULES` tuple in order and return the first matching rule's `RouteDecision`. A blank or `None` prompt returns `OUT_OF_SCOPE` with rule name `"empty"`. A prompt that matches no rule returns `OUT_OF_SCOPE` with rule name `"default:no-rule"`. The router **never guesses** — unmatched prompts are refused.

**The 5 rules in order (already defined in RULES tuple):**
1. `scope:write-request` → `OUT_OF_SCOPE` — transfer, wire, send money, move money, etc.
2. `keyword:policy` → `POLICY` — policy, policies, procedure, guideline, dispute
3. `keyword:eligibility` → `ELIGIBILITY` — eligible, eligibility, qualify, pre-approved, borrow
4. `keyword:transaction` → `TRANSACTION` — transaction, statement, activity, spent, recent, last N
5. `keyword:policy-topic` → `POLICY` — fees, limits, charges, overdraft, interest, terms, KYC

**Implementation:**
```python
def route(self, prompt: str | None) -> RouteDecision:
    if not prompt or not prompt.strip():
        return RouteDecision(intent=Intent.OUT_OF_SCOPE, rule="empty")
    for rule in RULES:
        if rule.pattern.search(prompt):
            return RouteDecision(intent=rule.intent, rule=rule.name)
    return RouteDecision(intent=Intent.OUT_OF_SCOPE, rule="default:no-rule")
```

**Verification (run from inside `python-orchestrator/`):**
```powershell
cd python-orchestrator
pip install -r requirements-dev.txt
pytest tests/test_router.py -q
```
*Expect:* All tests pass. `"transfer 500"` → `OUT_OF_SCOPE`. `"loan eligibility"` → `ELIGIBILITY`. `"last 5 transactions"` → `TRANSACTION`. `"overdraft fee"` → `POLICY` (rule `keyword:policy-topic`). Blank prompt → `OUT_OF_SCOPE` rule `empty`.

### Task 6 — CAP Freshness (`python-orchestrator/orchestrator/knowledge.py` + `reviews/CAP_TRADEOFFS.md`)
**Files to edit:**
1. `python-orchestrator/orchestrator/knowledge.py` — implement `PolicyKnowledgeBase.freshness()`
2. `reviews/CAP_TRADEOFFS.md` — fill in your CAP trade-off reasoning

**Objective — `freshness()` in `knowledge.py`:**
Compute how old the policy index is. `age = now − indexed_at` in whole seconds. `stale` is `True` only when `age > max_staleness` (exactly at the limit is still fresh). Return a `Freshness` dataclass.

**Implementation:**
```python
def freshness(self) -> Freshness:
    now = self._clock()
    age = now - self._indexed_at
    age_seconds = int(age.total_seconds())
    stale = age > self._max_staleness
    return Freshness(
        stale=stale,
        age_seconds=age_seconds,
        max_staleness_seconds=int(self._max_staleness.total_seconds()),
        indexed_at=self._indexed_at.isoformat(),
    )
```

**Objective — `reviews/CAP_TRADEOFFS.md`:**
The file has 4 questions. Answer each one explaining:
- Why core banking data (MCP) is `CONSISTENT_READ` (AP → give up A, pick CP for writes; CP for reads too)
- Why the policy index is `EVENTUAL` (served stale for reads, up to 24 h)
- What happens if the MCP server is unreachable (fail closed: 503, never stale answer)
- What a debit tool would change (must be CP — cannot allow double-debit under partition)

**Verification (run from inside `python-orchestrator/`):**
```powershell
cd python-orchestrator
pytest tests/test_knowledge.py -q
```
*Expect:* All tests pass. Fresh index → `stale=False`. Index 25 h old → `stale=True`. Exactly at 24 h → `stale=False`.

### Task 7 — Domain Specialists
**Files to edit:**
- `python-orchestrator/orchestrator/agents/eligibility.py` — `assess_personal_loan()` + `EligibilityAgent.run()`
- `python-orchestrator/orchestrator/agents/policy.py` — `PolicyAgent.run()`
- `python-orchestrator/orchestrator/agents/transaction.py` — `TransactionAgent.run()`

Also implement `PolicyKnowledgeBase.retrieve()` in `python-orchestrator/orchestrator/knowledge.py`.

**Eligibility — `assess_personal_loan(profile)` (Decimal arithmetic only):**
```python
# Rule order (first match wins):
# 1. Any loan not CURRENT → DECLINE
# 2. No bureauScore → REFER
# 3. score < 650 → DECLINE
# 4. 650 <= score < 700 → REFER
# 5. score >= 700: multiplier = 2 if score >= 740 else 1
#    headroom = multiplier * liquid - outstanding
#    headroom <= 0 → DECLINE, else ELIGIBLE with maxAdditionalCredit = headroom rounded down to cents
```
Asha Raman (10428): score 742, liquid 96740.18, outstanding 150000.00 → `ELIGIBLE`, `maxAdditionalCredit` = `43480.36`.

**Policy — `PolicyKnowledgeBase.retrieve(query, k=2)` in `knowledge.py`:**
```python
# score = len(terms(query) & doc_terms) / len(terms(query))
# Keep docs with score >= min_score (default 0.6), best k, ties by doc id
# Returns [] when no query terms or no doc above threshold
```

**Verification (run from inside `python-orchestrator/`):**
```powershell
cd python-orchestrator
pytest tests/test_agents.py -q
pytest tests/test_knowledge.py -q
```
*Expect:* Eligibility for 10428 → `ELIGIBLE`, answer contains `43480.36`. Transaction → answer contains `GROCERY`. Policy query `"overdraft fee"` → hits `POL-OVERDRAFT-01`.

### Task 8 — Orchestrator Pipeline (`orchestrator.py` + `resilience.py`)
**Files to edit:**
- `python-orchestrator/orchestrator/orchestrator.py` — `Orchestrator.handle(request)`
- `python-orchestrator/orchestrator/resilience.py` — `FallbackChain` and circuit breakers

**`Orchestrator.handle` — 5 steps:**
1. Route with `self._router.route(request.prompt)` using `time.perf_counter()` for timing
2. If `OUT_OF_SCOPE` or no agent registered → return `_refusal(decision, started, routed)`
3. Run the agent: `self._agents[decision.intent].run(request)` — let `CustomerNotFound` and `DataUnavailable` propagate (server maps them to 404/503)
4. If outcome needs a model call → ask `self._fallback.call(request, outcome)` once; path is `PRIMARY`, `SECONDARY`, or `TOOL_ONLY`
5. If outcome does not need a model (e.g. `UNGROUNDED`) → path is `outcome.path_hint`, not degraded, answer is `outcome.summary`
Return the full response dict with `timingsMs: {route, agent, model, total}`.

**Verification (run from inside `python-orchestrator/`):**
```powershell
cd python-orchestrator
pytest tests/test_orchestrator.py tests/test_resilience.py -q
```
*Expect:* All tests pass. `OUT_OF_SCOPE` → `agent=NONE`, `path=REFUSED`. Primary model down → `path=SECONDARY`, `degraded=True`. Both models down → `path=TOOL_ONLY`, `degraded=True`.

### Task 9 — Live End-to-End Check
**Objective:** Run both processes and issue live queries.

**Terminal 1 — Start Python Orchestrator (Port 8090):**
```powershell
cd python-orchestrator
pip install -r requirements.txt
$env:ORCH_MCP_URL = "http://127.0.0.1:8080/mcp"
python -m orchestrator
```

**Terminal 2 — Start Java Gateway (Port 8080):**
```powershell
mvn spring-boot:run
```

**Terminal 3 — Live Queries (PowerShell):**

⚠️ `X-RM-Id` is a **header**, not in the JSON body. `customerId` and `prompt` are the only body fields.
```powershell
# Policy — fee query → agent:POLICY, answer includes 250.00 / overdraft / fee
Invoke-RestMethod -Uri http://localhost:8080/v1/copilot/query -Method POST `
  -Headers @{Authorization="Bearer lab-token"; "X-RM-Id"="rm-14"} `
  -ContentType "application/json" `
  -Body '{"customerId":"10428","prompt":"What is the fee for an outward transfer?"}'

# Transaction — history query → agent:TRANSACTION, answer includes GROCERY
Invoke-RestMethod -Uri http://localhost:8080/v1/copilot/query -Method POST `
  -Headers @{Authorization="Bearer lab-token"; "X-RM-Id"="rm-14"} `
  -ContentType "application/json" `
  -Body '{"customerId":"10428","prompt":"Show the last transactions"}'

# Eligibility — loan query → agent:ELIGIBILITY, answer includes 742 and ELIGIBLE
Invoke-RestMethod -Uri http://localhost:8080/v1/copilot/query -Method POST `
  -Headers @{Authorization="Bearer lab-token"; "X-RM-Id"="rm-14"} `
  -ContentType "application/json" `
  -Body '{"customerId":"10428","prompt":"Is Asha eligible for a personal loan?"}'

# Out-of-scope — write request → agent:NONE, path:REFUSED
Invoke-RestMethod -Uri http://localhost:8080/v1/copilot/query -Method POST `
  -Headers @{Authorization="Bearer lab-token"; "X-RM-Id"="rm-14"} `
  -ContentType "application/json" `
  -Body '{"customerId":"10428","prompt":"Transfer 5000 to account 99"}'

# Auth test — missing bearer → HTTP 401 (Python never sees this)
Invoke-RestMethod -Uri http://localhost:8080/v1/copilot/query -Method POST `
  -Headers @{"X-RM-Id"="rm-14"} `
  -ContentType "application/json" `
  -Body '{"customerId":"10428","prompt":"Hello"}'
```

**Automated end-to-end check (optional):**
```powershell
cd python-orchestrator
python e2e_check.py
```
*Expect:* All 3 specialist routes return correct agents. 401 on missing bearer. `path=REFUSED` on write request.